# Day 15 — Native tool / function calling

Yesterday's Echo agent parsed JSON *we* invented. Production models have a **built-in**
way to call tools: you pass a list of tool **schemas**, and instead of plain text the
model replies with `tool_calls`. You run the tool and feed the result back.

**The loop:**
1. Send messages **+ tool schemas** → model.
2. If the reply has `tool_calls`: run each tool, append a `role: "tool"` result, loop.
3. If it has plain content: that's the final answer.

This is exactly what every framework (OpenAI SDK, AutoGen, LangGraph) does under the hood.
You'll use [`shared.llm.chat_with_tools`](../../shared/llm.py), which returns the full
assistant message so you can read `.tool_calls`.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run the cell. Stuck? The solution is below — but try first.

In [ ]:
import json
from shared.llm import chat_with_tools
from shared.tools import calculator

TOOLS = [{
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate basic arithmetic such as '2 + 3 * 4'.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
        },
    },
}]
REGISTRY = {"calculator": calculator}

def run(goal, max_steps=5):
    messages = [{"role": "user", "content": goal}]
    for _ in range(max_steps):
        msg = chat_with_tools(messages, TOOLS)
        # TODO 1: if msg.tool_calls is empty/None, we're done -> return msg.content
        # TODO 2: otherwise append the assistant msg (carry its tool_calls forward)
        # TODO 3: for each tc in msg.tool_calls: args = json.loads(tc.function.arguments or "{}");
        #         result = REGISTRY[tc.function.name](**args);
        #         append {"role": "tool", "tool_call_id": tc.id, "content": str(result)}
        raise NotImplementedError("complete the tool-calling loop")
    return "stopped (hit max_steps)"

# print(run("What is 12 * (3 + 4)? Use the calculator."))

## 🔒 Solution

One correct way to do it. Compare it with your version.

In [ ]:
import json
from shared.llm import chat_with_tools
from shared.tools import calculator

TOOLS = [{
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate basic arithmetic such as '2 + 3 * 4'.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
        },
    },
}]
REGISTRY = {"calculator": calculator}

def run(goal, max_steps=5):
    messages = [{"role": "user", "content": goal}]
    for _ in range(max_steps):
        msg = chat_with_tools(messages, TOOLS)
        if not msg.tool_calls:
            return msg.content
        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            fn = REGISTRY.get(tc.function.name)
            result = fn(**args) if fn else f"unknown tool: {tc.function.name}"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
    return "stopped (hit max_steps)"

print(run("What is 12 * (3 + 4)? Use the calculator."))